# Virtual worms and entropy triangles (CLasIC)

This notebook runs the population-level CLasIC workflow used in Chapter 5 of the thesis.

The steps are:

1. Load one concentration from the compiled `.mat` file of Lin et al. (2023).
2. Build the K virtual worms (single-presentation traces sampled **without replacement**, so no trace is used in more than one worm) and the averaged worm (their entry-wise mean, stored as worm 1).
3. Choose the K-FCA branch: `minplus` (activation, `R >= phi`, phi >= 0) or `maxplus` (inhibition, `R <= phi`, phi < 0).
4. Compute all formal concepts of every worm and the SMET and CMET tables across the phi sweep.
5. Visualize the four standard triangles.

All reusable logic lives in `src/`; the notebook only orchestrates the pipeline. The same tables can be produced from the command line with `python scripts/compute_clasic.py`, and the random references with `python scripts/compute_references.py`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpltern  # registers the ternary projection
from IPython.display import display

BASE_DIR = Path("..").resolve()
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from calculate_entropies import jentropies, build_smet_from_intents, build_smet_from_extents
from fca import build_extents_intents_all_worms
from utils import (
    build_thesis_worms,
    build_odor_neuron_structs,
    compute_peak_matrix,
    phi_grid,
    binarize_branch,
)
from visualize_triangles import (
    plot_smet_triangle,
    plot_smet_interactive,
    plot_cmet_triangle,
    plot_cmet_interactive,
)

## Configuration

- `concentration`: `4`, `5`, or `6` (dilutions 10^-4, 10^-5, 10^-6)
- `branch`: `"minplus"` (activation) or `"maxplus"` (inhibition)
- `seed`: seed of the virtual-worm sampling (42 in the thesis)
- `window_seconds`: half-width of the window around the peak (2 s)

In [ ]:
concentration = 4
branch = "minplus"
seed = 42
window_seconds = 2.0

In [ ]:
mat_path = BASE_DIR / "contexts" / f"compiled_data_conc_{concentration}.mat"
# Optional: contexts exported by the K-FCA live script of KFCA_for_neuraldata.
# If this folder exists its thresholds are used (exact reproduction of the thesis);
# otherwise every distinct entry of R(c) is used as a threshold.
context_dir = BASE_DIR / "contexts" / f"od_by_neuron{concentration}_23x11_{branch}"

worms = build_thesis_worms(mat_path, seed=seed, window_seconds=window_seconds)
data = worms["data"]
contexts_cont = worms["contexts"]          # index 0 = averaged worm, 1..K = virtual worms
odor_names, neuron_names = data["odor_names"], data["neuron_names"]

structs, n2i = build_odor_neuron_structs(data["compiled_LR_ordered"], neuron_names)
R = compute_peak_matrix(structs, n2i, window_seconds=window_seconds)   # graded matrix of Chapter 4
phi_values = phi_grid(branch, R=R, context_dir=context_dir)

summary_df = pd.DataFrame(
    {
        "item": ["concentration", "branch", "n_odors", "n_neurons", "K virtual worms", "contexts (K + averaged)", "n_phi"],
        "value": [concentration, branch, len(odor_names), len(neuron_names), worms["K"], contexts_cont.shape[0], len(phi_values)],
    }
)
display(summary_df)

## Thresholding

At every phi, each of the K + 1 graded contexts is binarized with the rule of the selected branch.

In [ ]:
def binary_contexts(phi):
    return binarize_branch(contexts_cont, phi, branch)

print("density of the averaged worm at the first and last phi:",
      binary_contexts(phi_values[0])[0].mean().round(3), binary_contexts(phi_values[-1])[0].mean().round(3))

## SMET for neurons across all `phi` values

This block reproduces the neuron-side SMET table using concept intents.


In [ ]:
SEDF_allphis = pd.concat(
    [build_smet_from_intents(binary_contexts(phi), odor_names=odor_names, neuron_names=neuron_names, phi=phi)
     for phi in phi_values],
    ignore_index=True,
)
display(SEDF_allphis.head())
# To save this table: SEDF_allphis.to_csv(BASE_DIR / "results" / f"SEDF_neurons_{100 + concentration}_{branch}.csv", index=False)

## SMET for odorants across all `phi` values

This is the odor-side SMET table built from concept extents.


In [ ]:
SEDF_allphis_odor = pd.concat(
    [build_smet_from_extents(binary_contexts(phi), odor_names=odor_names, neuron_names=neuron_names, phi=phi)
     for phi in phi_values],
    ignore_index=True,
)
display(SEDF_allphis_odor.head())
# To save this table: SEDF_allphis_odor.to_csv(BASE_DIR / "results" / f"SEDF_odorants_{100 + concentration}_{branch}.csv", index=False)

## CMET across all `phi` values

The channel table is computed from the extents and intents concatenated across worms.


In [ ]:
CMET_list = []
for phi in phi_values:
    ext_all, int_all = build_extents_intents_all_worms(binary_contexts(phi), odor_names=odor_names, neuron_names=neuron_names)
    cmet_phi = jentropies(ext_all, int_all)
    cmet_phi["phi"] = phi
    CMET_list.append(cmet_phi)

CMET_all_phi = pd.concat(CMET_list, ignore_index=True)
display(CMET_all_phi.head())
# To save this table: CMET_all_phi.to_csv(BASE_DIR / "results" / f"CMET_{100 + concentration}_{branch}.csv", index=False)

## Static four-panel view

This figure shows the four standard triangles across the full `phi` sweep for the selected concentration.

Each panel is generated from the same family of plotting functions, with only the dataframe and the triangle type changing. The goal is to display the full trajectory, not a single threshold snapshot.

### Parameters used in the plotting calls

- `df`: dataframe containing the values to plot. In this notebook, each panel receives the dataframe corresponding to one entropy representation.
- `mode`: only used in SMET plots. It controls how the dataframe is filtered before plotting:
  - `global_avg`: plots the pooled `@AVERAGE` trajectory across all virtual worms,
  - `per_worm_avg`: plots one average trajectory per worm,
  - `one_worm_all`: plots all rows for a selected worm.
- `typ`: only used in CMET plots. It selects which channel side is drawn:
  - `X`: source-side channel decomposition,
  - `Y`: destination-side channel decomposition,
  - `XY`: joint channel representation, when available.
- `phi_val`: if a numeric value is given, the plot is restricted to that single threshold. Here it is set to `None` so that the whole `phi` sweep can be displayed.
- `accumulate`: if `True`, all points up to the selected `phi` are shown. In this static figure it is used together with `phi_val=None`, so the complete trajectory is drawn.
- `interactive`: enables or disables widget-based interaction. It is set to `False` here because this is a static figure.
- `title`: title shown above each panel.
- `ax`: target ternary axis where the panel is drawn. This is what allows the four plots to be placed in a single figure.
- `fixed_phi_range`: if `True`, the color scale is forced to use the same `phi` range across panels. This makes the colors directly comparable.
- `phi_min`, `phi_max`: lower and upper bounds of the shared color scale.
- `add_colorbar`: controls whether each subplot adds its own colorbar. It is set to `False` here to keep the layout compact.
- `show_trajectory`: if `True`, a black trajectory is overlaid to make the path across `phi` easier to follow.


In [ ]:
phi_values = np.sort(SEDF_allphis["phi"].astype(float).unique())

fig = plt.figure(figsize=(20, 5.5))

ax1 = fig.add_subplot(1, 4, 1, projection="ternary", rotation=180)
plot_smet_triangle(
    df=SEDF_allphis_odor,
    mode="global_avg",
    phi_val=None,
    accumulate=True,
    interactive=False,
    title="SMET odorants | all phi values",
    ax=ax1,
    fixed_phi_range=True,
    phi_min=phi_values.min(),
    phi_max=phi_values.max(),
    add_colorbar=False,
    show_trajectory=True,
)

ax2 = fig.add_subplot(1, 4, 2, projection="ternary")
plot_cmet_triangle(
    df=CMET_all_phi,
    typ="X",
    phi_val=None,
    accumulate=True,
    interactive=False,
    title="CMET X | all phi values",
    ax=ax2,
    fixed_phi_range=True,
    phi_min=phi_values.min(),
    phi_max=phi_values.max(),
    add_colorbar=False,
    show_trajectory=True,
)

ax3 = fig.add_subplot(1, 4, 3, projection="ternary")
plot_cmet_triangle(
    df=CMET_all_phi,
    typ="Y",
    phi_val=None,
    accumulate=True,
    interactive=False,
    title="CMET Y | all phi values",
    ax=ax3,
    fixed_phi_range=True,
    phi_min=phi_values.min(),
    phi_max=phi_values.max(),
    add_colorbar=False,
    show_trajectory=True,
)

ax4 = fig.add_subplot(1, 4, 4, projection="ternary", rotation=180)
plot_smet_triangle(
    df=SEDF_allphis,
    mode="global_avg",
    phi_val=None,
    accumulate=True,
    interactive=False,
    title="SMET neurons | all phi values",
    ax=ax4,
    fixed_phi_range=True,
    phi_min=phi_values.min(),
    phi_max=phi_values.max(),
    add_colorbar=False,
    show_trajectory=True,
)

plt.tight_layout()
plt.show()
# To save this figure, use fig.savefig(BASE_DIR / "figures" / f"all_triang_{concentration}_{branch}.png", dpi=300, bbox_inches="tight").

## Interactive viewers and display options

The SMET plotting functions support three display modes:

- `global_avg`: plots only the `@AVERAGE` row pooled across all virtual worms. This is the cleanest view to follow the global trajectory across `phi`.
- `per_worm_avg`: plots one `@AVERAGE` trajectory per worm (worm 1 is the averaged worm). This is useful to compare worm-to-worm variability while keeping the representation compact.
- `one_worm_all`: plots all individual entries for a single worm; `worm_ids=1` gives the individual neuron/odorant trajectories of the averaged worm used in the thesis. This is the most detailed view and can optionally be restricted to one or several names.

For CMET, the main distinction is not the aggregation mode but the channel side being plotted:

- `X`: source-side channel decomposition
- `Y`: destination-side channel decomposition
- `XY`: joint channel representation, when available in the dataframe



The `accumulate` option changes the temporal interpretation of the slider:
- if `True`, all points up to the selected `phi` are shown,
- if `False`, only the exact selected `phi` is shown.

In [ ]:

plot_smet_interactive(
    df=SEDF_allphis,
    phi_values=sorted(SEDF_allphis["phi"].unique()),
    mode="global_avg",
    title="SMET neurons",
)
# To save one view from this widget, reproduce the same parameters in a static plotting cell and save the resulting figure in BASE_DIR / "figures".


In [ ]:

plot_smet_interactive(
    df=SEDF_allphis_odor,
    phi_values=sorted(SEDF_allphis_odor["phi"].unique()),
    mode="global_avg",
    title="SMET odorants",
)
# To save one view from this widget, reproduce the same parameters in a static plotting cell and save the resulting figure in BASE_DIR / "figures".


In [ ]:

plot_cmet_interactive(
    df=CMET_all_phi,
    phi_values=sorted(CMET_all_phi["phi"].unique()),
)
# To save one view from this widget, reproduce the same parameters in a static plotting cell and save the resulting figure in BASE_DIR / "figures".
